# Learnings 04: Runbook And Operational Commands

This notebook is the practical runbook for this project.

Its goal is simple: if we want to run the project again later, test it, ingest papers, query it, use Telegram, or debug health issues, this notebook should tell us exactly what to do.

This notebook focuses on real terminal commands, what each command does, and when we should use it.

## 1. What This Project Runs

This project is a multi-service system. It is not just one Python script.

The important services are:

- `rag-api`: FastAPI backend on port `8000`
- `rag-airflow`: Airflow on port `8081`
- `rag-postgres`: PostgreSQL on port `5435`
- `rag-opensearch`: OpenSearch on port `9202`
- `rag-redis`: Redis on port `6382`
- `rag-ollama`: Ollama on port `11435`
- `rag-telegram-bot`: Telegram bot runner
- `rag-langfuse-web`: Langfuse UI on port `3002`

Important meaning:

- the API is what we query directly using curl or a client
- Airflow helps run ingestion and indexing workflows
- PostgreSQL stores paper metadata and parsed text
- OpenSearch stores searchable chunks
- Ollama generates answers
- Redis caches repeated exact queries
- Telegram gives a chat interface
- Langfuse gives tracing and observability

## 2. First Step: Start The Stack

Run this from the project root:

```bash
docker compose up -d postgres redis opensearch ollama api airflow telegram-bot langfuse-web langfuse-worker
```

What this does:

- starts the main storage, search, model, API, workflow, Telegram, and tracing services
- runs them in detached mode because of `-d`
- uses the definitions from `compose.yml`

If we only want the core app without Telegram or Langfuse, we can start a smaller set:

```bash
docker compose up -d postgres redis opensearch ollama api airflow
```

If we made code changes and want to rebuild images too:

```bash
docker compose up -d --build api airflow telegram-bot
```

What this does:

- rebuilds the selected services from the latest source code
- then starts them

## 3. Check If Services Are Running

Useful command:

```bash
docker ps --format '{{.Names}}\t{{.Status}}\t{{.Ports}}'
```

What this does:

- shows running containers
- shows their status
- shows which host ports are mapped

To check a single service:

```bash
docker ps --filter name=rag-api
docker ps --filter name=rag-airflow
docker ps --filter name=rag-opensearch
docker ps --filter name=rag-postgres
docker ps --filter name=rag-telegram-bot
docker ps --filter name=rag-langfuse-web
```

Use these when we want to confirm that a specific service is alive.

## 4. Health Checks

### API health

```bash
curl http://localhost:8000/api/v1/health
```

What this does:

- confirms the API is alive
- checks database connectivity
- checks OpenSearch connectivity
- checks Ollama connectivity

### Verify the OpenSearch index state from Airflow context

```bash
docker compose exec airflow python -c "
from dags.arxiv_ingestion.indexing import verify_hybrid_index
print(verify_hybrid_index())
"
```

What this does:

- checks the hybrid index
- reports total chunk count
- reports unique paper count
- reports average chunks per paper

This is one of the most important validation commands in the project.

## 5. Check Logs

These are the most useful log commands.

### API logs

```bash
docker logs --tail 100 rag-api
```

Use this when:

- `/ask` fails
- `/ask-agentic` fails
- health looks strange
- Langfuse tracing seems missing

### Airflow logs

```bash
docker logs --tail 100 rag-airflow
```

Use this when:

- ingestion fails
- indexing fails
- Airflow startup is unhealthy

### Telegram bot logs

```bash
docker logs --tail 100 rag-telegram-bot
```

Use this when:

- the bot does not respond
- polling fails
- Telegram configuration is suspected

### Langfuse web logs

```bash
docker logs --tail 100 rag-langfuse-web
```

Use this when:

- Langfuse UI is not opening
- sign-in issues appear
- tracing setup feels broken

### OpenSearch logs

```bash
docker logs --tail 100 rag-opensearch
```

Use this when:

- indexing is failing
- search is failing
- there are disk watermark or read-only issues

## 6. How To Run Ingestion Manually

This is the main command we used for manual ingestion:

```bash
docker compose exec airflow python -c "
import asyncio
from dags.arxiv_ingestion.fetching import run_paper_ingestion_pipeline
results = asyncio.run(run_paper_ingestion_pipeline(target_date='20260429', process_pdfs=True))
print('Papers fetched:', results['papers_fetched'])
print('PDFs parsed:', results['pdfs_parsed'])
print('Errors:', len(results['errors']))
print('First errors:', results['errors'][:5])
"
```

What this does:

- runs the ingestion pipeline manually inside the Airflow container
- fetches papers for one date
- downloads PDFs
- parses PDFs
- stores metadata and parsed content in PostgreSQL

Important note:

- `target_date` must be in `YYYYMMDD` format
- example: `20260429`

### Can we ingest papers from multiple dates or months?

Yes.

The manual ingestion helper takes a single date per run, but the database is not limited to one date. That means we can run ingestion multiple times for different dates and build up a larger collection over time.

Meaning:

- one run fetches one day
- many runs can build a multi-day or multi-month collection
- indexing can then include all parsed papers already stored in PostgreSQL

So the current collection is not fundamentally one-day only. It was just that we tested one specific date and got 15 papers.

### Important behavior to remember

The scheduled Airflow daily workflow is designed around one target date at a time.

This means the project is naturally shaped around targeted daily ingestion.

A good mental model is:

- one ingestion run normally targets one date
- that one run adds its papers into PostgreSQL
- later indexing reads all available parsed papers and pushes chunks into OpenSearch
- if we keep doing this over many dates, the collection accumulates over time

So the project supports a growing corpus even though the helper itself is single-date.

But manual/CLI usage lets us grow the corpus across dates by running it repeatedly.

## 7. How To Build A Larger Multi-Date Collection

If we want more than one day of papers, we can run ingestion on several dates.

Example pattern:

```bash
docker compose exec airflow python -c "
import asyncio
from dags.arxiv_ingestion.fetching import run_paper_ingestion_pipeline
for d in ['20260427', '20260428', '20260429']:
    results = asyncio.run(run_paper_ingestion_pipeline(target_date=d, process_pdfs=True))
    print(d, results['papers_fetched'], results['pdfs_parsed'], len(results['errors']))
"
```

What this does:

- loops over multiple dates
- fetches/stores papers for each date
- accumulates them in PostgreSQL

This is the safest and most understandable backfill method with the current project design because:

- each date is handled separately
- failures are easier to inspect per day
- reruns are easier if one day fails
- it matches the current targeted daily ingestion pattern

### Example: collect papers for a longer period such as the last 3 months

If we want something like "download papers from the current date back to the last 3 months", the practical way with the current codebase is to generate a list of dates and run the single-day helper for each date.

Example pattern:

```bash
docker compose exec airflow python -c "
import asyncio
from datetime import datetime, timedelta
from dags.arxiv_ingestion.fetching import run_paper_ingestion_pipeline

end_date = datetime.strptime('20260429', '%Y%m%d')
start_date = end_date - timedelta(days=90)

current = start_date
while current <= end_date:
    d = current.strftime('%Y%m%d')
    results = asyncio.run(run_paper_ingestion_pipeline(target_date=d, process_pdfs=True))
    print(d, results['papers_fetched'], results['pdfs_parsed'], len(results['errors']))
    current += timedelta(days=1)
"
```

What this does:

- computes a 90-day range
- runs one ingestion pass per day
- stores all successful papers in PostgreSQL
- builds a multi-date corpus gradually

### Important note about result size

The current config uses a default `max_results` value from the arXiv settings. During our project work, that default was small for testing purposes, which is why a one-day run gave around 15 papers.

That means:

- the system is not limited to 15 papers forever
- but one run only fetches up to the configured limit
- if we want a bigger historical collection, we may want to raise `max_results` in config before doing a large backfill

### Advanced note: direct date-range support exists below the helper level

The lower-level ingestion logic supports `from_date` and `to_date`, but the main helper we used in operations is currently designed for one target date at a time.

So for normal project usage, the recommended method is still:

- run one date at a time in a loop
- then re-index once after the backfill is complete

After that, run indexing again so OpenSearch includes the larger collection.

Important caution:

- if we do not clear PostgreSQL and OpenSearch, the collection grows over time
- this is good when we want a bigger corpus
- this is not good if we wanted a clean one-date-only experiment

## 8. How To Run Indexing Manually

Main indexing command:

```bash
docker compose exec airflow python -c "
from dags.arxiv_ingestion.indexing import index_papers_hybrid
print(index_papers_hybrid())
"
```

What this does:

- reads parsed papers from PostgreSQL
- chunks the text
- generates Jina embeddings
- writes chunks into OpenSearch

Very important project-specific detail:

- for manual CLI runs, the indexing code now indexes all papers that already have parsed text
- so it is not limited to only the latest one-day fetch

That is why this works nicely after multiple-date ingestion runs.

### Verify indexing after running it

```bash
docker compose exec airflow python -c "
from dags.arxiv_ingestion.indexing import verify_hybrid_index
print(verify_hybrid_index())
"
```

Use this after each meaningful ingestion/indexing cycle.

## 9. How To Run Gradio

Gradio is the lightweight interactive UI for the project.

Important idea:

- Gradio is a client interface
- FastAPI is still the main backend
- Gradio talks to the API at `http://localhost:8000/api/v1`
- so the API must already be running before Gradio is useful

### Start the backend first

```bash
docker compose up -d postgres redis opensearch ollama api airflow
```

Then verify the backend:

```bash
curl http://localhost:8000/api/v1/health
```

### Start Gradio

```bash
uv run python gradio_launcher.py
```

What this does:

- starts the Gradio web interface
- uses the local Python environment
- sends user questions to the FastAPI backend

### Which port does Gradio use?

In this project, Gradio is configured to run on port `7861`.

Meaning:

- API uses `8000`
- Gradio uses `7861`
- different services use different ports on the same machine

If Gradio says the port is already busy, then some other process is already using `7861`.

### Why `localhost` can fail in the browser

This is a remote-machine concept.

If the terminal is running on a remote server such as `gpusrv000`, but the browser is running on your laptop, then:

- `localhost` in the terminal means the remote server
- `localhost` in the browser means your own laptop

So if Gradio is running on the remote server, opening `http://localhost:7861` directly in your laptop browser will fail unless you bridge that port.

### Why we use SSH port forwarding before opening localhost

SSH port forwarding makes a port on the remote server appear locally on your laptop.

That is why we do an SSH tunnel first and then open `localhost` in the browser.

### Command for SSH port forwarding

Run this from your local machine terminal:

```bash
ssh -L 7861:localhost:7861 ayush_ldap@gpusrv000
```

What this means:

- local laptop port `7861`
- forwards to remote server port `7861`
- through the SSH connection to `gpusrv000`

After that tunnel is active, open this in your local browser:

```text
http://localhost:7861
```

That is the reason it works: the SSH tunnel maps the remote Gradio service into your local machine.

### Practical sequence for Gradio when working on a remote server

1. Start backend services on the server.
2. Start Gradio on the server.
3. Open a local terminal on your laptop.
4. Create the SSH tunnel.
5. Open `http://localhost:7861` in your laptop browser.

### If Gradio does not open

Check these things:

- is the API healthy?
- is Gradio still running in the terminal?
- is port `7861` already occupied?
- is the browser on the same machine or a different one?
- if the browser is local and the server is remote, is the SSH tunnel active?


## 10. How To Ask Questions Through The API

### Hybrid search only

```bash
curl -X POST http://localhost:8000/api/v1/hybrid-search/ \
  -H "Content-Type: application/json" \
  -d '{
    "query": "self-evolving software agents",
    "size": 3,
    "use_hybrid": true
  }'
```

What this does:

- performs retrieval only
- does not generate the final answer
- useful for debugging search quality

### Normal RAG ask

```bash
curl -X POST http://localhost:8000/api/v1/ask \
  -H "Content-Type: application/json" \
  -d '{
    "query": "How do the indexed papers approach adaptive or evolving software agents?",
    "top_k": 5,
    "use_hybrid": true,
    "model": "qwen2.5:7b"
  }'
```

What this does:

- performs retrieval
- builds context
- asks Ollama to generate a grounded answer

### Streaming answer

```bash
curl -N -X POST http://localhost:8000/api/v1/stream \
  -H "Content-Type: application/json" \
  -d '{
    "query": "Summarize the key findings of the indexed papers on agent systems.",
    "top_k": 5,
    "use_hybrid": true,
    "model": "qwen2.5:7b"
  }'
```

What this does:

- streams the answer gradually
- useful when we want token-by-token style visibility

### Agentic ask

```bash
curl -X POST http://localhost:8000/api/v1/ask-agentic \
  -H "Content-Type: application/json" \
  -d '{
    "query": "How do the indexed papers approach adaptive or evolving software agents?",
    "top_k": 5,
    "use_hybrid": true,
    "model": "qwen2.5:7b"
  }'
```

What this does:

- runs the graph-based agentic RAG pipeline
- performs retrieval and grading
- returns reasoning metadata and trace ID


## 11. How To Use The Telegram Bot

If the Telegram bot is configured and running, usage is simple.

### Start the Telegram service if needed

```bash
docker compose up -d telegram-bot
```

### Check bot logs

```bash
docker logs --tail 100 rag-telegram-bot
```

### How to ask questions in Telegram

1. Open Telegram
2. Search for your bot username
3. Open the chat
4. Send `/start`
5. Ask your question in natural language

Example questions:

- `How do the indexed papers approach adaptive or evolving software agents?`
- `Summarize the main idea of self-evolving software agents.`
- `What papers in the collection discuss agent systems?`

What the Telegram bot does behind the scenes:

- receives the question
- performs retrieval
- calls the LLM
- returns a user-friendly answer in chat

## 12. How To Use Langfuse

Langfuse UI is available at:

- `http://localhost:3002` if the browser is on the same machine
- or use the server hostname/IP if browsing from another machine

Start Langfuse services if needed:

```bash
docker compose up -d langfuse-web langfuse-worker
```

Check logs:

```bash
docker logs --tail 100 rag-langfuse-web
docker logs --tail 100 rag-langfuse-worker
```

Why we use Langfuse:

- inspect traces
- compare `/ask` and `/ask-agentic`
- inspect prompts and responses
- debug quality problems later

## 13. When We Want A Fresh Experiment Versus A Growing Corpus

This is an important operational idea.

### If we want a growing collection

Do this:

- keep PostgreSQL data
- keep OpenSearch data
- run ingestion for more dates
- rerun indexing

Result:

- the collection grows beyond one day
- PostgreSQL holds papers from many ingestion runs
- OpenSearch can be refreshed to include the larger corpus

This is what we mean by targeted daily ingestion and accumulation over time.

The system does not automatically mean "one date forever".
Instead it means:

- the unit of ingestion is usually one date
- the lifetime of the collection can span many dates or months
- growth happens by repeating daily-style ingestion across many dates

So a practical long-term flow is:

1. ingest date A
2. ingest date B
3. ingest date C
4. keep stored papers in PostgreSQL
5. run indexing
6. query the larger combined collection

### If we want a clean experiment

Do this only when we intentionally want to reset the dataset:

- clear PostgreSQL paper rows
- clear or recreate the OpenSearch index
- then ingest and index again

Meaning:

- a one-day experiment is possible
- a multi-date or multi-month research collection is also possible

The project supports both styles depending on whether we preserve or reset stored data.

## 14. A Good Standard Workflow To Follow Later

If we return to this project later and want to use it normally, this is a good sequence:

1. Start the stack with Docker Compose.
2. Run the API health check.
3. Verify the index state.
4. If needed, ingest one or more target dates.
5. Run indexing.
6. Verify indexing again.
7. Test hybrid search.
8. Test `/ask`.
9. Test `/ask-agentic`.
10. Ask the same or similar question through Telegram if needed.
11. Use Langfuse to inspect traces when debugging or comparing behavior.

This gives a full repeatable operational cycle.

## 15. Final Practical Summary

The most important idea to remember is this:

- the project is already working end to end
- we know how to start it
- we know how to ingest papers
- we know how to index them
- we know how to query through curl
- we know how to query through Telegram
- we know how to inspect traces in Langfuse

And regarding paper coverage:

- the test run had around 15 papers for one date
- but the system is not limited to one date forever
- we can build a larger collection by ingesting more dates and re-indexing
- the current operational style is targeted daily ingestion plus long-term accumulation
- if we want a multi-month corpus, the practical method is to loop over dates and then re-index
- if we keep the database and search index, the corpus continues to grow over time

This notebook is the operational memory of the project.